# Mamba from scratch, for speech people

**CAS 784, Speech Recognition in Humans and Machines** · Bart Kowal & Xiao Shao · Nov 13, 2026

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bartkw12/mamba-speech-demo/blob/main/mamba_speech_demo.ipynb)

This notebook builds the core of **Mamba** (Gu & Dao, 2023) in plain PyTorch, so it runs anywhere,
even without a GPU. No special CUDA kernels, no installs.

| Part | What you'll see | Autopsy card fields | Time |
| --- | --- | --- | --- |
| 0 | Setup | | 30 s |
| 1 | A selective scan in ~20 lines, with sliders | Computations, Time | instant |
| 2 | Race: attention vs scan as the input gets longer | Time | ~1 min |
| 3 | Why selection matters: a "remember the name" task | Computations, Learning | ~1–2 min |
| 4 | Tiny Mamba vs tiny Transformer on spoken digits | Input, Output, Success, Learning | ~1–3 min |
| 5 | Look inside: what does the model write to memory? | Intermediate representations, Ambiguity, Failure modes | instant |
| 6 | Optional: a real pretrained Mamba language model | Modularity and context | ~2 min |
| 7 | Model autopsy card | all | |

**How to use it**
- Run cells top to bottom with **Shift + Enter**. Run Part 0 first.
- A GPU is optional. To use one: *Runtime → Change runtime type → T4 GPU*.
- This is a read-only copy. To keep your changes: *File → Save a copy in Drive*.
- Look for **🔮 Predict** (guess before you run) and **🧪 Try** (things to change).

Color code (same as the slides): <span style="color:#5BC0EB">**blue = input**</span>,
<span style="color:#7BC47F">**green = state / memory**</span>, <span style="color:#E8C766">**gold = attention**</span>,
<span style="color:#F28C38">**orange = selective (input-dependent)**</span>.

## Part 0 · Setup
Imports, picks a GPU if you have one, and defines the colors. Takes a few seconds.

In [ ]:
import math, time, random, os, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatLogSlider, IntSlider

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0); random.seed(0)

BLUE, GREEN, GOLD, ORANGE, GRAY = "#5BC0EB", "#7BC47F", "#E8C766", "#F28C38", "#888888"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

print(f"PyTorch {torch.__version__} · running on: {device.upper()}")
if device == "cpu":
    print("No GPU: that's fine, everything still runs (a bit slower in Parts 3 and 4).")

## Part 1 · A selective scan in ~20 lines

A state space model keeps a **state** `h` (a fixed-size memory) and updates it once per input step:

$$h_t = \bar A_t \, h_{t-1} + \bar B_t \, x_t \qquad y_t = C_t \, h_t$$

with $\bar A_t = e^{\Delta_t A}$ (how much old memory to **keep**) and $\bar B_t$ (how much of the new input to **write**).

* In a plain SSM, $\Delta, B, C$ are the same at every step.
* In **Mamba** they are computed from the current input: that's the *selective* part.

Here is the whole thing as a loop. Read it once, it's the heart of Mamba.

In [ ]:
def selective_scan_loop(x, delta, A, B, C):
    """
    x:     (L, D)  input: L time steps, D channels          (blue)
    delta: (L, D)  step size for every step and channel    (orange: depends on input in Mamba)
    A:     (D, N)  negative numbers: how fast memory fades
    B, C:  (L, N)  how input is written / how memory is read (orange)
    Returns y: (L, D) outputs and h_all: (L, D, N) every state along the way (green)
    """
    L, D = x.shape
    N = A.shape[1]
    h = torch.zeros(D, N)                                # the state starts empty
    ys, h_all = [], []
    for t in range(L):                                   # one small update per step: O(L)
        A_bar = torch.exp(delta[t, :, None] * A)         # in (0, 1): fraction of old memory kept
        B_bar = (A_bar - 1) / A * B[t][None, :]          # how much of x_t gets written (zero-order hold)
        h = A_bar * h + B_bar * x[t, :, None]            # update the memory
        ys.append(h @ C[t])                              # read the memory out
        h_all.append(h.clone())
    return torch.stack(ys), torch.stack(h_all)

### 🧪 Try: be the state

A toy "utterance" of 60 frames: mostly **filler** (silence, "um", noise) and one **key** frame at step 12 (think: *"my name is **Bond**"*).
One channel, one state number, $A=-1$, $B=C=1$.

You control $\Delta$ for filler frames and for the key frame separately:

* **Plain SSM:** set both sliders to the same value. Can you find a setting that remembers the key until the end *and* ignores the filler?
* **Selective (Mamba):** small $\Delta$ on filler, large $\Delta$ on the key. What happens to the memory?

🔮 **Predict first:** with both sliders at 0.5, how much of the key is left at step 60?

In [ ]:
L_toy, KEY_POS = 60, 12
g = torch.Generator().manual_seed(1)
x_toy = 0.35 * torch.randn(L_toy, 1, generator=g)       # filler: small random values
x_toy[KEY_POS] = 2.0                                    # the key frame
is_key = torch.zeros(L_toy, dtype=torch.bool); is_key[KEY_POS] = True

def show_state(delta_filler=0.5, delta_key=0.5):
    delta = torch.where(is_key, torch.tensor(delta_key), torch.tensor(delta_filler))[:, None]
    A = -torch.ones(1, 1); B = torch.ones(L_toy, 1); C = torch.ones(L_toy, 1)
    y, h = selective_scan_loop(x_toy, delta, A, B, C)
    fig, ax = plt.subplots(2, 1, figsize=(9, 4.2), sharex=True)
    ax[0].bar(range(L_toy), x_toy[:, 0], color=[ORANGE if k else BLUE for k in is_key])
    ax[0].set_ylabel("input x"); ax[0].set_title("Input: filler (blue) and one key frame (orange)")
    ax[1].plot(h[:, 0, 0], color=GREEN, lw=2.5)
    ax[1].axhline(0, color=GRAY, lw=0.8); ax[1].set_ylim(-1.2, 2.2)
    ax[1].set_ylabel("state h"); ax[1].set_xlabel("time step")
    kept = h[-1, 0, 0].item()
    selective = "plain SSM (same Δ everywhere)" if delta_filler == delta_key else "selective (Δ depends on input)"
    ax[1].set_title(f"Memory at the last step: {kept:+.2f}  (key value was +2.00) · {selective}")
    plt.tight_layout(); plt.show()

interact(show_state,
         delta_filler=FloatLogSlider(value=0.5, base=10, min=-3, max=1, step=0.1, description="Δ filler"),
         delta_key=FloatLogSlider(value=0.5, base=10, min=-3, max=1, step=0.1, description="Δ key"));

**What to notice**
* Large $\Delta$ = *reset and focus on this input*. Small $\Delta$ = *ignore this input, keep the memory*.
* With one shared $\Delta$ you must trade off: either the filler overwrites the key, or the key barely registers.
* Mamba learns a small network that outputs $\Delta_t, B_t, C_t$ from $x_t$, so it can do both. (Try: Δ filler = 0.01, Δ key = 10.)

📝 *Autopsy card → **Computations, Time**: what is computed per frame, and how does the past influence the present?*

## Part 2 · Race: attention vs the scan

Attention builds an $L \times L$ grid (every frame compares itself with every other frame). The scan carries a fixed-size state.

A Python `for` loop is slow, so real implementations use a **parallel scan**: the update $h_t = a_t h_{t-1} + b_t$ can be
solved in $\log_2 L$ levels of vectorized operations (same idea as a parallel prefix sum). Xiao's part explains this.
Below: a 15-line version, checked against the loop.

In [ ]:
def pscan(a, b):
    """Solve h_t = a_t * h_{t-1} + b_t (with h_0 = 0) along dim 1.
    Trick: two steps of the recurrence are again one step of the same form, so solve every 2nd step
    with a half-length scan (recursion), then fill in the steps in between. log2(L) levels, O(L) work."""
    L = a.shape[1]
    if L == 1:
        return b
    if L % 2:                                                    # pad to even length with a=1, b=0
        a = torch.cat([a, torch.ones_like(a[:, :1])], 1)
        b = torch.cat([b, torch.zeros_like(b[:, :1])], 1)
    a_e, a_o, b_e, b_o = a[:, 0::2], a[:, 1::2], b[:, 0::2], b[:, 1::2]
    h_o = pscan(a_o * a_e, a_o * b_e + b_o)                      # every 2nd state, via a half-length scan
    h_prev = torch.cat([torch.zeros_like(h_o[:, :1]), h_o[:, :-1]], 1)
    h_e = a_e * h_prev + b_e                                     # fill in the states in between
    return torch.stack([h_e, h_o], 2).flatten(1, 2)[:, :L]

def selective_scan(x, delta, A, B, C):
    """Batched, parallel version of selective_scan_loop.
    x, delta: (batch, L, D) · A: (D, N) · B, C: (batch, L, N) → y: (batch, L, D)"""
    A_bar = torch.exp(delta.unsqueeze(-1) * A)                       # (b, L, D, N)
    B_x = delta.unsqueeze(-1) * B.unsqueeze(2) * x.unsqueeze(-1)     # (b, L, D, N); Mamba's simplified B_bar = Δ·B
    h = pscan(A_bar, B_x)                                            # all states at once
    return (h @ C.unsqueeze(-1)).squeeze(-1)                         # read out with C_t

# Sanity check: parallel scan == simple loop
a, b = torch.rand(2, 100, 3), torch.randn(2, 100, 3)
h_loop, h = [], torch.zeros(2, 3)
for t in range(100):
    h = a[:, t] * h + b[:, t]; h_loop.append(h)
print("parallel scan matches the loop:", torch.allclose(pscan(a, b), torch.stack(h_loop, 1), atol=1e-5))

### 🔮 Predict, then run

We time one attention layer (naive: it really builds the grid) and one selective scan layer, both with 64 channels,
for sequences from 256 frames (5 s of audio at 50 frames/s) up to 30–60k frames (10–20 minutes). If the grid won't fit in memory, we skip attention.

**At what length does the scan become faster than attention?** Write your guess in the poll, then run.

In [ ]:
def naive_attention(q, k, v):
    scores = q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1])    # the L x L grid
    return torch.softmax(scores, dim=-1) @ v

def time_it(fn, reps=3):
    fn()                                                         # warm-up
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps): fn()
    if device == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / reps

D, N = 64, 16
max_L = 65536 if device == "cuda" else 32768
lengths = [2 ** k for k in range(8, int(math.log2(max_L)) + 1)]
# memory we allow the attention grid to use (it needs the scores and the softmax: 2 copies)
budget = 0.7 * torch.cuda.get_device_properties(0).total_memory if device == "cuda" else 3e9
A = -torch.rand(D, N, device=device) - 0.5
results = []
with torch.no_grad():
    for L in lengths:
        x = torch.randn(1, L, D, device=device)
        grid_gb = L * L * 4 / 1e9
        if 2 * grid_gb * 1e9 > budget:
            t_att, att_txt = float("nan"), f"too big: the grid alone needs {grid_gb:5.1f} GB"
        else:
            t_att = time_it(lambda: naive_attention(x, x, x))
            att_txt = f"{t_att*1e3:8.1f} ms (grid: {grid_gb*1e3:7.1f} MB)"
        delta = F.softplus(torch.randn(1, L, D, device=device))
        Bm, Cm = torch.randn(1, L, N, device=device), torch.randn(1, L, N, device=device)
        t_ssm = time_it(lambda: selective_scan(x, delta, A, Bm, Cm))
        results.append((L, t_att, t_ssm))
        print(f"L = {L:6d} frames ({L/50/60:5.1f} min of audio) · attention {att_txt} · scan {t_ssm*1e3:7.1f} ms")
        del x, delta, Bm, Cm
        if device == "cuda": torch.cuda.empty_cache()

Ls, ta, ts = map(np.array, zip(*results))
plt.figure(figsize=(7, 4))
plt.loglog(Ls, ta * 1e3, "o-", color=GOLD, lw=2.5, label="attention (builds L×L grid)")
plt.loglog(Ls, ts * 1e3, "o-", color=GREEN, lw=2.5, label="selective scan (fixed-size state)")
for L in Ls[np.isnan(ta)]:
    plt.annotate("attention:\nout of memory", (L, ts[Ls == L][0] * 1e3), xytext=(0, 25), textcoords="offset points",
                 ha="center", color="#C0392B", fontsize=8, arrowprops=dict(arrowstyle="-", color="#C0392B"))
plt.xlabel("sequence length L (frames)"); plt.ylabel("time per layer (ms)")
plt.title(f"One layer, 64 channels, on {device.upper()}"); plt.legend(); plt.grid(alpha=0.3, which="both")
plt.show()
print(f"For 1 hour of audio (180,000 frames) the attention grid alone would need "
      f"{180_000**2 * 4 / 1e9:,.0f} GB per head, per layer.")

**What to notice**
* On a log-log plot, attention's line is steeper (slope ≈ 2: double L, 4× the work). The scan is ≈ linear.
* For short inputs attention can still win: matrix multiplies are extremely well optimized. The crossover depends on the hardware.
* Our scan is a simple teaching version. The real Mamba kernel is far faster (Xiao's part).

📝 *Autopsy card → **Time**: how does cost grow with the length of the input?*

## The Mamba block, in plain PyTorch

This is the block from Gu & Dao (Figure 3): expand → short convolution → SiLU → **selective SSM**, gated by a second branch,
then project back. `selective=False` turns off selection: $\Delta, B, C$ become fixed learned numbers (a plain SSM),
everything else stays the same. We'll use that switch in Part 3.

In [ ]:
class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, expand=2, d_conv=4, selective=True):
        super().__init__()
        d_inner = expand * d_model
        self.selective, self.d_state = selective, d_state
        self.in_proj = nn.Linear(d_model, 2 * d_inner)                   # main branch + gate branch
        self.conv = nn.Conv1d(d_inner, d_inner, d_conv, groups=d_inner, padding=d_conv - 1)
        # Δ starts between 0.001 and 0.1 (as in Mamba): stored as the inverse of softplus
        dt0 = torch.exp(torch.empty(d_inner).uniform_(math.log(1e-3), math.log(1e-1)))
        dt_bias = dt0 + torch.log(-torch.expm1(-dt0))
        if selective:                                                    # Δ, B, C computed from the input
            self.dt_rank = math.ceil(d_model / 16)
            self.x_proj = nn.Linear(d_inner, self.dt_rank + 2 * d_state, bias=False)
            self.dt_proj = nn.Linear(self.dt_rank, d_inner)
            with torch.no_grad(): self.dt_proj.bias.copy_(dt_bias)
        else:                                                            # Δ, B, C fixed (plain SSM)
            self.B = nn.Parameter(torch.randn(d_state) / math.sqrt(d_state))
            self.C = nn.Parameter(torch.randn(d_state) / math.sqrt(d_state))
            self.dt = nn.Parameter(dt_bias)
        # A: negative, different decay rate per state dimension (S4D-Real initialisation)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1).float()).repeat(d_inner, 1))
        self.D = nn.Parameter(torch.ones(d_inner))                       # skip connection
        self.out_proj = nn.Linear(d_inner, d_model)
        self.last_delta = None                                           # saved for Part 5

    def forward(self, x):                                                # x: (batch, L, d_model)
        b, L, _ = x.shape
        u, z = self.in_proj(x).chunk(2, dim=-1)
        u = F.silu(self.conv(u.transpose(1, 2))[..., :L].transpose(1, 2))   # causal short conv
        A = -torch.exp(self.A_log)
        if self.selective:
            dt, B, C = self.x_proj(u).split([self.dt_rank, self.d_state, self.d_state], dim=-1)
            delta = F.softplus(self.dt_proj(dt))                         # (b, L, d_inner)
        else:
            delta = F.softplus(self.dt).expand(b, L, -1)
            B = self.B.expand(b, L, -1); C = self.C.expand(b, L, -1)
        self.last_delta = delta.detach()
        y = selective_scan(u, delta, A, B, C) + u * self.D
        return self.out_proj(y * F.silu(z))                              # gate, project back

class MambaLayer(nn.Module):
    """Pre-norm residual wrapper. bidirectional=True adds a second block that reads the sequence backwards
    (the 'external bidirectional' idea from Zhang et al. 2025)."""
    def __init__(self, d_model, bidirectional=False, **kw):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.fwd = MambaBlock(d_model, **kw)
        self.bwd = MambaBlock(d_model, **kw) if bidirectional else None
    def forward(self, x):
        h = self.norm(x)
        out = self.fwd(h)
        if self.bwd is not None:
            out = out + self.bwd(h.flip(1)).flip(1)
        return x + out

def n_params(model): return sum(p.numel() for p in model.parameters())
print("MambaBlock ready. One block with d_model=64 has", f"{n_params(MambaBlock(64)):,}", "parameters.")

## Part 3 · Why selection matters: "remember the name"

Each sequence is 64 random tokens (filler). Somewhere in it is the marker `NAME` followed by a name token.
At the end comes `?` and the model must output **the name**. Sometimes the name is mentioned twice;
then the **latest** one counts (*"my name is Ada … actually, it's Bond"*).

```
 7 3 12 NAME 5 9 0 14 ... 2 NAME 11 6 ... 8 ?   →  answer: 11
```

This is a version of the *selective copying* task from Gu & Dao (Figure 2). We train two identical 2-layer models,
one with selection on and one with it off. Chance is 1/16 = 6%.

🔮 **Predict:** which one gets above 90%?

In [ ]:
VOCAB, NAME, QUERY, SEQ_LEN = 16, 16, 17, 64

def make_batch(batch_size, L=SEQ_LEN):
    seq = torch.randint(0, VOCAB, (batch_size, L))
    target = torch.empty(batch_size, dtype=torch.long)
    for i in range(batch_size):
        n_mentions = random.choice([1, 2])
        positions = sorted(random.sample(range(0, L - 3, 2), n_mentions))
        for p in positions:
            seq[i, p] = NAME
            seq[i, p + 1] = random.randrange(VOCAB)
        target[i] = seq[i, positions[-1] + 1]
    seq[:, -1] = QUERY
    return seq.to(device), target.to(device)

class TokenModel(nn.Module):
    def __init__(self, selective, d_model=32, n_layers=2):
        super().__init__()
        self.emb = nn.Embedding(VOCAB + 2, d_model)
        self.layers = nn.ModuleList(MambaLayer(d_model, selective=selective, d_state=16) for _ in range(n_layers))
        self.norm, self.head = nn.LayerNorm(d_model), nn.Linear(d_model, VOCAB)
    def forward(self, seq):
        x = self.emb(seq)
        for layer in self.layers: x = layer(x)
        return self.head(self.norm(x[:, -1]))          # answer at the "?" position

def train_tokens(selective, steps=400, batch_size=64, lr=3e-3):
    torch.manual_seed(0); random.seed(0)
    model = TokenModel(selective).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    accs, t0 = [], time.time()
    for step in range(1, steps + 1):
        seq, target = make_batch(batch_size)
        loss = F.cross_entropy(model(seq), target)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if step % 25 == 0:
            with torch.no_grad():
                seq, target = make_batch(256)
                accs.append((step, (model(seq).argmax(-1) == target).float().mean().item()))
    name = "selective (Mamba)" if selective else "plain SSM"
    print(f"{name:18s}: final accuracy {accs[-1][1]:5.1%} · {n_params(model):,} parameters · {time.time()-t0:.0f} s")
    return model, accs

model_sel, acc_sel = train_tokens(selective=True)
model_lti, acc_lti = train_tokens(selective=False)

plt.figure(figsize=(7, 3.8))
for accs, color, label in [(acc_sel, ORANGE, "selective SSM (Mamba)"), (acc_lti, GREEN, "plain SSM (same Δ, B, C everywhere)")]:
    plt.plot(*zip(*accs), color=color, lw=2.5, label=label)
plt.axhline(1 / VOCAB, color=GRAY, ls="--", lw=1, label="chance")
plt.ylim(0, 1.02); plt.xlabel("training step"); plt.ylabel("accuracy"); plt.legend(loc="center right")
plt.title("Remember the (latest) name"); plt.show()

**What to notice**
* Same architecture, same size, same training. The only difference is whether $\Delta, B, C$ depend on the input.
* The plain SSM can only blur the past with a fixed decay, so it can't tell *the name* from the filler around it, or reset when a newer name arrives.
* 🧪 **Try:** make the sequence longer (`SEQ_LEN = 256`) or allow up to 3 mentions. Which model suffers more?

📝 *Autopsy card → **Computations, Learning**: what does the model have to learn to solve this?*

## Part 4 · Tiny Mamba vs tiny Transformer on spoken digits

The [Free Spoken Digit Dataset](https://github.com/Jakobovski/free-spoken-digit-dataset) has 3,000 recordings of
the digits 0–9 (6 speakers, 8 kHz). We turn each clip into **log-mel frames** (40 mel bands, 10 ms hop: 100 frames per second)
and train two small classifiers:

* **Mamba:** 2 bidirectional Mamba layers, 32 channels
* **Transformer:** 2 Transformer encoder layers, 32 channels, 4 attention heads

Both average their outputs over time and predict the digit. Standard split: recordings 0–4 of every speaker/digit are the test set.

In [ ]:
# Download the dataset (about 40 MB) and compute log-mel features
if not os.path.exists("fsdd"):
    os.system("git clone -q --depth 1 https://github.com/Jakobovski/free-spoken-digit-dataset fsdd")
from scipy.io import wavfile

SR, N_FFT, HOP, N_MELS, MAX_FRAMES = 8000, 256, 80, 40, 100

def mel_filterbank(sr=SR, n_fft=N_FFT, n_mels=N_MELS):
    mel = lambda f: 2595 * np.log10(1 + f / 700)
    inv = lambda m: 700 * (10 ** (m / 2595) - 1)
    pts = inv(np.linspace(mel(0), mel(sr / 2), n_mels + 2))
    bins = np.floor((n_fft + 1) * pts / sr).astype(int)
    fb = np.zeros((n_mels, n_fft // 2 + 1))
    for m in range(1, n_mels + 1):
        l, c, r = bins[m - 1], bins[m], bins[m + 1]
        fb[m - 1, l:c] = (np.arange(l, c) - l) / max(c - l, 1)
        fb[m - 1, c:r] = (r - np.arange(c, r)) / max(r - c, 1)
    return torch.tensor(fb, dtype=torch.float32)

FB = mel_filterbank()
def log_mel(wave):
    spec = torch.stft(torch.tensor(wave), N_FFT, HOP, win_length=200, window=torch.hann_window(200),
                      return_complex=True).abs() ** 2
    return torch.log(FB @ spec + 1e-6).T                     # (frames, 40)

X, lengths, labels, is_test, waves = [], [], [], [], []
for path in sorted(glob.glob("fsdd/recordings/*.wav")):
    digit, speaker, idx = os.path.basename(path)[:-4].split("_")
    sr, wave = wavfile.read(path)
    wave = wave.astype(np.float32)
    wave /= np.abs(wave).max() + 1e-9
    feats = log_mel(wave)[:MAX_FRAMES]
    feats = (feats - feats.mean(0)) / (feats.std(0) + 1e-5)   # normalise each clip
    T = feats.shape[0]
    X.append(F.pad(feats, (0, 0, 0, MAX_FRAMES - T))); lengths.append(T)
    labels.append(int(digit)); is_test.append(int(idx) < 5); waves.append(wave)

X, lengths, labels, is_test = torch.stack(X), torch.tensor(lengths), torch.tensor(labels), torch.tensor(is_test)
print(f"{len(X)} clips · {(~is_test).sum().item()} train / {is_test.sum().item()} test · "
      f"feature tensor {tuple(X.shape)} = (clips, frames, mel bands)")

i = 0
plt.figure(figsize=(7, 2.6))
plt.imshow(X[i, :lengths[i]].T, origin="lower", aspect="auto", cmap="magma")
plt.title(f"Input to both models: log-mel frames of a spoken '{labels[i].item()}'"); plt.xlabel("frame (10 ms)"); plt.ylabel("mel band")
plt.show()

In [ ]:
def masked_mean(h, lengths):
    mask = (torch.arange(h.shape[1], device=h.device)[None, :] < lengths[:, None]).float().unsqueeze(-1)
    return (h * mask).sum(1) / mask.sum(1)

class MambaClassifier(nn.Module):
    def __init__(self, d_model=32, n_layers=2, bidirectional=True):
        super().__init__()
        self.inp = nn.Linear(N_MELS, d_model)
        self.layers = nn.ModuleList(MambaLayer(d_model, bidirectional=bidirectional, d_state=16) for _ in range(n_layers))
        self.norm, self.head = nn.LayerNorm(d_model), nn.Linear(d_model, 10)
    def forward(self, x, lengths):
        h = self.inp(x)
        for layer in self.layers: h = layer(h)
        return self.head(masked_mean(self.norm(h), lengths))

class TransformerClassifier(nn.Module):
    def __init__(self, d_model=32, n_layers=2, n_heads=4):
        super().__init__()
        self.inp = nn.Linear(N_MELS, d_model)
        self.pos = nn.Parameter(0.02 * torch.randn(1, MAX_FRAMES, d_model))     # learned positions
        layer = nn.TransformerEncoderLayer(d_model, n_heads, 4 * d_model, dropout=0.1, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.norm, self.head = nn.LayerNorm(d_model), nn.Linear(d_model, 10)
    def forward(self, x, lengths):
        pad = torch.arange(x.shape[1], device=x.device)[None, :] >= lengths[:, None]
        h = self.enc(self.inp(x) + self.pos[:, :x.shape[1]], src_key_padding_mask=pad)
        return self.head(masked_mean(self.norm(h), lengths))

def train_digits(model, epochs=10, batch_size=64, lr=2e-3):
    torch.manual_seed(0)
    model = model.to(device)
    tr = torch.where(~is_test)[0]; te = torch.where(is_test)[0]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(tr) / batch_size))
    t0 = time.time()
    # batches of similar-length clips, trimmed to their longest clip (much less padding to process)
    tr_sorted = tr[torch.argsort(lengths[tr] + torch.rand(len(tr)))]
    batches = list(tr_sorted.split(batch_size))
    for ep in range(epochs):
        model.train()
        for k in torch.randperm(len(batches)).tolist():
            b = batches[k]; T = lengths[b].max()
            loss = F.cross_entropy(model(X[b, :T].to(device), lengths[b].to(device)), labels[b].to(device))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    train_time = time.time() - t0
    model.eval()
    with torch.no_grad():
        preds = torch.cat([model(X[b].to(device), lengths[b].to(device)).argmax(-1).cpu() for b in te.split(256)])
    acc = (preds == labels[te]).float().mean().item()
    return dict(model=model, acc=acc, preds=preds, params=n_params(model), time=train_time)

runs = {}
for name, make in [("Mamba (bidirectional)", lambda: MambaClassifier(bidirectional=True)),
                   ("Transformer", lambda: TransformerClassifier())]:
    runs[name] = train_digits(make())
    r = runs[name]
    print(f"{name:22s} test accuracy {r['acc']:6.1%} · {r['params']:,} parameters · trained in {r['time']:.0f} s")

**What to notice**
* Both reach roughly the same high accuracy on this small, clean task (93–96% in our test runs; it varies a little run to run). Mamba doesn't need attention to recognize a short word.
* Our teaching Mamba trains *slower* than the Transformer here: PyTorch's attention is heavily optimized, our scan is not, and
  these clips are short (under 100 frames), where attention is cheap anyway. The official CUDA kernel closes that gap.
* 🧪 **Try:** train `MambaClassifier(bidirectional=False)` (reads left to right only, like a streaming model). Does accuracy change?
  On a task this small the difference may be within noise. Zhang et al. (2025) study this at full scale (Xiao's part).
* 🧪 **Try:** make the models smaller (`d_model=16`) or train for fewer epochs. Which one degrades first?

📝 *Autopsy card → **Input, Output, Success, Learning**.*

In [ ]:
# 🧪 Try it: a left-to-right (unidirectional) Mamba. Uncomment and run (~1–2 min).
# runs["Mamba (left-to-right)"] = train_digits(MambaClassifier(bidirectional=False))
# print(f"left-to-right Mamba test accuracy: {runs['Mamba (left-to-right)']['acc']:.1%}")

## Part 5 · Look inside: what does Mamba write to memory?

Recall: large $\Delta$ = *write this frame into memory*, small $\Delta$ = *mostly ignore it*.
Below, for one test clip, we plot the spectrogram and the average $\Delta$ of the first Mamba layer (left-to-right block) at every frame.

🔮 **Predict:** will $\Delta$ be larger during the spoken word or during the silence around it?

In [ ]:
mamba = runs["Mamba (bidirectional)"]["model"]
test_idx = torch.where(is_test)[0]

def look_inside(k=0):
    i = test_idx[k].item(); T = lengths[i].item()
    with torch.no_grad():
        pred = mamba(X[i:i+1].to(device), lengths[i:i+1].to(device)).argmax(-1).item()
    delta = mamba.layers[0].fwd.last_delta[0, :T].mean(-1).cpu()
    w = np.pad(waves[i], (0, max(0, T * HOP - len(waves[i]))))[: T * HOP]
    energy = torch.tensor(w).reshape(T, HOP).pow(2).mean(-1)
    fig, ax = plt.subplots(2, 1, figsize=(8, 4.2), sharex=True)
    ax[0].imshow(log_mel(waves[i])[:T].T, origin="lower", aspect="auto", cmap="magma")   # un-normalised, easier to read
    ax[0].set_title(f"Spoken '{labels[i].item()}' · model says '{pred}' " + ("✓" if pred == labels[i].item() else "✗"))
    ax[0].set_ylabel("mel band")
    ax[1].plot(delta, color=ORANGE, lw=2.5, label="mean Δ, layer 1")
    ax[1].set_ylabel("Δ (write strength)"); ax[1].set_xlabel("frame (10 ms)")
    ax2 = ax[1].twinx(); ax2.fill_between(range(T), energy / energy.max(), color=BLUE, alpha=0.2)
    ax2.set_yticks([]); ax2.set_ylabel("loudness", color=BLUE)
    ax[1].legend(loc="upper right"); plt.tight_layout(); plt.show()

interact(look_inside, k=IntSlider(value=0, min=0, max=len(test_idx) - 1, description="test clip"));

In [ ]:
# Where does the Transformer or Mamba go wrong? Confusion matrices and a few mistakes you can listen to.
from IPython.display import Audio, display
fig, axes = plt.subplots(1, len(runs), figsize=(4.2 * len(runs), 3.8))
for ax, (name, r) in zip(np.atleast_1d(axes), runs.items()):
    cm = torch.zeros(10, 10, dtype=torch.long)
    for t, p in zip(labels[is_test], r["preds"]): cm[t, p] += 1
    ax.imshow(cm, cmap="Greens"); ax.set_title(f"{name}\n{r['acc']:.1%} correct")
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_xticks(range(10)); ax.set_yticks(range(10))
    for a in range(10):
        for b in range(10):
            if cm[a, b] and a != b: ax.text(b, a, cm[a, b].item(), ha="center", va="center", color="red", fontsize=9)
plt.tight_layout(); plt.show()

wrong = [j for j, (t, p) in enumerate(zip(labels[is_test], runs["Mamba (bidirectional)"]["preds"])) if t != p]
print(f"Mamba got {len(wrong)} test clips wrong. Listen to a few:")
for j in wrong[:3]:
    i = test_idx[j].item()
    print(f"  true '{labels[i].item()}', predicted '{runs['Mamba (bidirectional)']['preds'][j].item()}'")
    display(Audio(waves[i], rate=SR))

**Discuss**
* Does $\Delta$ line up with the word, the silence, or the onsets? There is no guaranteed answer: this is the autopsy.
* Which digits get confused? Are those confusions ones a human listener might also make (*ambiguity*), or are they model failures?

📝 *Autopsy card → **Intermediate representations, Ambiguity, Failure modes**.*

## Part 6 · Optional (take-home): a real pretrained Mamba

`state-spaces/mamba-130m-hf` is a 130M-parameter Mamba language model from the authors, loadable with Hugging Face
`transformers`. Without the CUDA kernels it uses a slower pure-PyTorch path, which is fine for a few sentences.
Downloads about 500 MB.

In [ ]:
RUN_PART_6 = False   # set to True to run
if RUN_PART_6:
    from transformers import AutoTokenizer, MambaForCausalLM
    tok = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
    lm = MambaForCausalLM.from_pretrained("state-spaces/mamba-130m-hf").to(device)
    cfg = lm.config
    print(f"{cfg.num_hidden_layers} Mamba blocks · d_model {cfg.hidden_size} · inner size {cfg.intermediate_size} · state size N = {cfg.state_size}")
    print(f"Memory carried between tokens: {cfg.num_hidden_layers} × {cfg.intermediate_size} × {cfg.state_size} = "
          f"{cfg.num_hidden_layers * cfg.intermediate_size * cfg.state_size:,} numbers, no matter how long the text is.")
    ids = tok("Speech recognition in humans and machines", return_tensors="pt").input_ids.to(device)
    out = lm.generate(ids, max_new_tokens=40, do_sample=True, top_p=0.9, temperature=0.8)
    print("\n" + tok.decode(out[0]))

## Part 7 · Model autopsy card: Mamba

Fill this in with your neighbor (double-click a cell to edit it). The hints point to the parts of this notebook that help.

### Input
<details><summary>Hint</summary>What does the model take in? In Part 4: log-mel frames. In Zhang et al.: speech features at 16 kHz. In general?</details>

✍️ *Your answer:*

### Output
<details><summary>Hint</summary>What comes out, per frame or per sequence?</details>

✍️ *Your answer:*

### Success
<details><summary>Hint</summary>How is it judged? Accuracy in Part 4, word error rate in Zhang et al. Compared with what?</details>

✍️ *Your answer:*

### Intermediate representations
<details><summary>Hint</summary>What is the state h, and what does it summarise? What did Δ look like in Part 5?</details>

✍️ *Your answer:*

### Computations
<details><summary>Hint</summary>What happens at each step? Which parts are linear, which nonlinear? What does selection add (Part 3)?</details>

✍️ *Your answer:*

### Time
<details><summary>Hint</summary>How is the past carried forward? How does cost grow with length (Part 2)? Can it run in real time?</details>

✍️ *Your answer:*

### Learning
<details><summary>Hint</summary>What is trained, on what data, with what objective?</details>

✍️ *Your answer:*

### Modularity and context
<details><summary>Hint</summary>Is Mamba a model or a building block? How much context can it use, and how?</details>

✍️ *Your answer:*

### Ambiguity
<details><summary>Hint</summary>How could later context resolve an ambiguous frame? What changes with bidirectional Mamba?</details>

✍️ *Your answer:*

### Failure modes
<details><summary>Hint</summary>Where should it fail? Think: exact recall from far back, fixed-size memory, the confusions in Part 5.</details>

✍️ *Your answer:*

---
**Sources.** Gu & Dao (2023), *Mamba: Linear-Time Sequence Modeling with Selective State Spaces*, arXiv:2312.00752 ·
Zhang et al. (2025), *Mamba in Speech: Towards an Alternative to Self-Attention*, IEEE/ACM TASLP 33 ·
Free Spoken Digit Dataset (Jakobovski et al., CC BY-SA 4.0). The pure-PyTorch block here is a simplified teaching version,
not the official implementation.